In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import os
import pickle
from pathlib import Path

import pandas as pd
import scanpy as sc

# Xenium probes

In [ ]:
# TCRseq information
adata = sc.read_h5ad("data/xenium/processed/08.1-kidney_tcr_clonal_clusters.h5ad")

In [ ]:
av = [v for v in adata.var_names if v.startswith("TRAV")]
bv = [v for v in adata.var_names if v.startswith("TRBV")]

In [ ]:
bvmap = {
    "TRBV10": ["TRBV10-1", "TRBV10-2", "TRBV10-3"],
    "TRBV11": ["TRBV11-1", "TRBV11-2"],
    "TRBV12": ["TRBV12-3", "TRBV12-4", "TRBV12-5"],
    "TRBV18_19": ["TRBV18", "TRBV19"],
    "TRBV4": ["TRBV4-1", "TRBV4-2"],
    "TRBV5-6": ["TRBV5-3", "TRBV5-5", "TRBV5-6", "TRBV5-7"],
    "TRBV6": ["TRBV6-1", "TRBV6-2", "TRBV6-5", "TRBV6-7"],
    "TRBV7-2_3": ["TRBV7-2", "TRBV7-3"],
}
avmap = {
    "TRAV12": ["TRAV12-1", "TRAV12-2", "TRAV12-3"],
    "TRAV14DV4": ["TRAV14/DV4"],
    "TRAV19_21": ["TRAV19", "TRAV21"],
    "TRAV23DV6": ["TRAV23/DV6"],
    "TRAV29DV5": ["TRAV29/DV5"],
    "TRAV36DV7": ["TRAV36/DV7"],
    "TRAV8-4": ["TRAV8-2", "TRAV8-3", "TRAV8-4", "TRAV8-5", "TRAV8-6"],
    "TRAV9": ["TRAV9-1", "TRAV9-2"],
}

In [ ]:
def replacing_vnames(df, col, mapdic):
    flattened_dict = {
        old_val: new_key for new_key, old_vals in mapdic.items() for old_val in old_vals
    }
    df[col] = df[col].replace(flattened_dict)
    return df

In [ ]:
del adata

# Load for P126K

In [ ]:
ref_dir = "/bonn-epyc/projects/cedric/spatialTCR/kidney_data/refdata"

In [ ]:
df = pd.read_csv(f"{ref_dir}/P126K_Sobj_New_META.csv")
df.dropna(inplace=True)
cd4 = df[df.seurat_clusters.isin([0, 1, 3])]
cd8 = df[df.seurat_clusters == 2]

In [ ]:
def extract_AV_BV(df):
    srs = df["CTgene"]
    av = srs.apply(lambda s: str(s).split("_")[0].split(".")[0], 1)
    bv = srs.apply(lambda s: str(s).split("_")[1].split(".")[0], 1)
    df["AV"] = av
    df["BV"] = bv
    return df


cd4 = extract_AV_BV(cd4)
cd8 = extract_AV_BV(cd8)

In [ ]:
cd4 = replacing_vnames(cd4, "AV", avmap)
cd4 = replacing_vnames(cd4, "BV", bvmap)
cd8 = replacing_vnames(cd8, "AV", avmap)
cd8 = replacing_vnames(cd8, "BV", bvmap)

In [ ]:
cd4["celltype"] = "CD4+"
cd8["celltype"] = "CD8+"
df = pd.concat([cd4, cd8], axis=0)
df = df[["AV", "BV", "celltype"]]
df.columns = ["TRA", "TRB", "celltype"]
with open("data/tcr-seq/processed/refdic_p126.pkl", "wb") as handle:
    pickle.dump(df, handle)

# Load rest patients

In [ ]:
path = f"{ref_dir}/TCRseq/TCR_ANCA_kidney_Cedric/"
files = [path + x + "/outs/filtered_contig_annotations.csv" for x in os.listdir(path)]

In [ ]:
# Summarize TCRseq of patients
DF = pd.DataFrame()
for file in files:
    df = pd.read_csv(file)
    print(df.shape)
    df = df[df["barcode"].map(df["barcode"].value_counts()) == 2]
    dfnew = df.pivot(index="barcode", columns="chain", values="v_gene").reset_index()
    del df
    dfnew.index.name = None
    # sample folder is parent of outs/ (.../P126K_TCR/outs/...)
    dfnew["sample"] = Path(file).parts[-3]
    DF = pd.concat([DF, dfnew], axis=0).reset_index(drop=True)

In [ ]:
# Get refdic and Leiden Clusterings
refdf = pd.read_csv(f"{ref_dir}/ref_spatiotemp_robin.csv", index_col="Unnamed: 0")
refdic = {}
for sample in DF["sample"].unique():
    if sample == "P138K_TCR":
        continue
    sDF = DF[DF["sample"] == sample]
    ixs = list(sDF.barcode)
    srefdf = refdf[refdf["sample"] == sample.split("_")[0]].copy()
    refidx = [x.split("-P")[0] for x in srefdf.index]
    print(sample, set(ixs).intersection(refidx).__len__())
    srefdf = srefdf.loc[[f"{x}-{sample.split('_')[0]}" for x in refidx]]
    if srefdf.empty:
        continue
    srefdf = srefdf.dropna(how="all", axis=1)
    refdic[sample.split("_")[0]] = srefdf[["sample", "leiden_res0.2"]]
    # print(refdf.loc[ixs].shape,len(ixs))
# Rename clustername to phenotype
for key in refdic.keys():
    refdic[key].replace([0, 5], "CD4+", inplace=True)
    refdic[key].replace([1, 7], "CD8+", inplace=True)

In [ ]:
# with open('refdfdic.pkl', 'wb') as handle:
#     pickle.dump(refdic, handle)
for key in refdic.keys():
    sDF = DF[DF["sample"] == key + "_TCR"]
    srs = refdic[key]["leiden_res0.2"]
    # srs.name = "celltype"
    srs.index = [x.split("-P")[0] for x in srs.index]
    sDF["celltype"] = sDF["barcode"].map(srs)
    refdic[key] = sDF

In [ ]:
# Rename TRAV and TRBV unify with Xenium notation
for key in refdic.keys():
    df = refdic[key]
    df = replacing_vnames(df, "TRA", avmap)
    df = replacing_vnames(df, "TRB", bvmap)
    refdic[key] = df
correctedDF = pd.DataFrame()
for k, df in refdic.items():
    correctedDF = pd.concat([correctedDF, df], axis=0)

In [ ]:
refav = correctedDF["TRA"].unique()
refbv = correctedDF["TRB"].unique()

In [ ]:
# Keeping the genes in scref, which overlaps with Xenium av and bv
for key in refdic.keys():
    df = refdic[key]
    df = df[df["TRA"].isin(av)]
    df = df[df["TRB"].isin(bv)]
    refdic[key] = df

In [ ]:
with open("data/tcr-seq/processed/refdfdic_alignednaming.pkl", "wb") as handle:
    pickle.dump(refdic, handle)